<a href="https://colab.research.google.com/github/samrat-dhakal-11/AgenticAi-by-LangGraph/blob/main/postgresql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-openai langchain-tavily openai

from google.colab import drive
drive.mount("/content/drive")

from google.colab import userdata
from langchain_tavily import TavilySearch
from openai import OpenAI
from langchain_openai import ChatOpenAI

key = userdata.get("OPENAI_API_KEY")
tkey = userdata.get("TAVILY_API_KEY")
nurl = userdata.get("NEON_URL")
search_tool = TavilySearch(tavily_api_key=tkey, max_results=10,search_depth="advanced")
llm = ChatOpenAI(
    model="gpt-5.4-mini",
    api_key=key
)

client=OpenAI(api_key=key)

In [ ]:
import psycopg2

In [ ]:
conn=psycopg2.connect(nurl)
cur=conn.cursor()

In [ ]:
conn.rollback()

In [ ]:
cur.execute("""
CREATE TABLE IF NOT EXISTS samrat(
  id SERIAL PRIMARY KEY,
  user_email TEXT NOT NULL,
  complain TEXT NOT NULL,
  status TEXT DEFAULT 'pending')
""")

In [ ]:
conn.commit()

In [ ]:
cur.execute("SELECT * FROM practice_complaints")
print(cur.fetchall())

[(3, 'sita@gmail.com', 'Damaged box', 'closed'), (1, 'yoyo@gmail.com', 'Late delivery', 'reviewed')]


In [ ]:

cur.execute("""
CREATE TABLE IF NOT EXISTS practice_users (
    email TEXT PRIMARY KEY,
    name TEXT
)
""")
cur.execute("INSERT INTO practice_users (email, name) VALUES (%s, %s) ON CONFLICT (email) DO NOTHING",
            ("yoyo@gmail.com", "Yoyo"))
cur.execute("INSERT INTO practice_users (email, name) VALUES (%s, %s) ON CONFLICT (email) DO NOTHING",
            ("ram@gmail.com", "Ram"))
conn.commit()

In [ ]:

# 1. Inner join: only complaints with a matching user
cur.execute("""
SELECT practice_users.name, practice_complaints.complain, practice_complaints.status
FROM practice_complaints
JOIN practice_users ON practice_complaints.user_email = practice_users.email
""")
print(cur.fetchall())

# 2. Left join: keep every complaint
cur.execute("""
SELECT practice_users.name, practice_complaints.complain, practice_complaints.status
FROM practice_complaints
LEFT JOIN practice_users ON practice_complaints.user_email = practice_users.email
""")
print(cur.fetchall())

[('Yoyo', 'Late delivery', 'reviewed')]
[(None, 'Damaged box', 'closed'), ('Yoyo', 'Late delivery', 'reviewed')]
